# Algoritmo Z

El array Z de una cadena $s$ guarda en $Z_i$ la longitud del prefijo común más largo entre $s$ y $s[i:]$ (por convenio, $Z_0 = n$). Se calcula en $O(n)$ manteniendo la caja Z $[l, r)$: el tramo más a la derecha que coincide con un prefijo de $s$. Si $i$ cae dentro, $s[i:r]$ es una copia de $s[i-l:r-l]$, de modo que

$$Z_i \ge \min(r - i,\ Z_{i-l}) \qquad \text{si } l < i < r$$

y solo hay que seguir comparando a partir de ahí. Cada comparación con éxito hace avanzar $r$, que nunca retrocede, así que el total es lineal. Con él se buscan patrones en $O(n + m)$, igual que con KMP.

## 1. Array Z

Calcular el array Z, mostrarlo alineado bajo la cadena y comprobarlo contra la definición, que cuesta $O(n^2)$.

In [1]:
import random
import timeit

def array_z(s):
    n = len(s)
    z, l, r = [0] * n, 0, 0
    if n:
        z[0] = n
    for i in range(1, n):
        if i < r:
            z[i] = min(r - i, z[i - l])
        while i + z[i] < n and s[z[i]] == s[i + z[i]]:
            z[i] += 1
        if i + z[i] > r:
            l, r = i, i + z[i]
    return z

def array_z_bruto(s):
    def prefijo_comun(i):
        k = 0
        while i + k < len(s) and s[k] == s[i + k]:
            k += 1
        return k
    return [prefijo_comun(i) for i in range(len(s))]

s = "aabxaabxcaabxaabxay"
print(" ".join(f"{c:>2}" for c in s))
print(" ".join(f"{v:>2}" for v in array_z(s)))

random.seed(1)
for _ in range(1000):
    t = "".join(random.choices("ab", k=random.randint(0, 20)))
    assert array_z(t) == array_z_bruto(t)

 a  a  b  x  a  a  b  x  c  a  a  b  x  a  a  b  x  a  y
19  1  0  0  4  1  0  0  0  8  1  0  0  5  1  0  0  1  0


## 2. Búsqueda de patrón

Calcular el array Z de `patron + separador + texto`, con un separador que no aparezca en ninguno de los dos: las posiciones con $Z_i = m$ son apariciones del patrón. Cuesta $O(n + m)$. Comprobar contra la búsqueda ingenua.

In [2]:
def busca_z(texto, patron, separador="\0"):
    m = len(patron)
    z = array_z(patron + separador + texto)
    return [i - m - 1 for i in range(m + 1, len(z)) if z[i] == m]

def ingenua(texto, patron):
    m = len(patron)
    return [i for i in range(len(texto) - m + 1) if texto[i:i + m] == patron]

for _ in range(1000):
    t = "".join(random.choices("ab", k=30))
    p = "".join(random.choices("ab", k=random.randint(1, 4)))
    assert busca_z(t, p) == ingenua(t, p)
print(busca_z("abracadabra", "abra"), busca_z("AAAAA", "AA"))

[0, 7] [0, 1, 2, 3]


## 3. Período mínimo

El menor $p$ con $p + Z_p = n$ es el período mínimo de $s$: la cadena coincide consigo misma desplazada $p$ posiciones. Si además $p$ divide a $n$, $s$ es una repetición de su prefijo de longitud $p$. Calcularlo y comprobarlo con fuerza bruta.

In [3]:
def periodo_z(s):
    z = array_z(s)
    return next((p for p in range(1, len(s)) if p + z[p] == len(s)), len(s))

def periodo_bruto(s):
    n = len(s)
    return next(p for p in range(1, n + 1) if all(s[i] == s[i + p] for i in range(n - p)))

for _ in range(1000):
    t = "".join(random.choices("ab", k=random.randint(1, 14)))
    assert periodo_z(t) == periodo_bruto(t)

for t in ("abcabcabc", "abcabcab", "aaaa", "abcd"):
    p = periodo_z(t)
    repeticion = f"= {t[:p]!r} × {len(t) // p}" if len(t) % p == 0 and p < len(t) else ""
    print(f"{t:10} período {p} {repeticion}")

abcabcabc  período 3 = 'abc' × 3
abcabcab   período 3 
aaaa       período 1 = 'a' × 4
abcd       período 4 


## 4. Bordes de una cadena

Un borde es un prefijo propio que también es sufijo. La posición $i$ empieza un sufijo que coincide con el prefijo de longitud $n - i$ justo cuando $i + Z_i = n$. Listar todos los bordes con el array Z y comprobarlo con fuerza bruta.

In [4]:
def bordes(s):
    z = array_z(s)
    return sorted(len(s) - i for i in range(1, len(s)) if i + z[i] == len(s))

for _ in range(1000):
    t = "".join(random.choices("ab", k=random.randint(1, 14)))
    assert bordes(t) == [k for k in range(1, len(t)) if t[:k] == t[-k:]]
print([("abacaba"[:k]) for k in bordes("abacaba")], bordes("aaaa"))

['a', 'aba'] [1, 2, 3]


## 5. Comparación con KMP

El array Z y la tabla de fallos de KMP contienen la misma información: cada $Z_i > 0$ dice que el prefijo de longitud $j + 1$ es borde del prefijo que acaba en $i + j$, para $j < Z_i$. Convertir el array Z en la tabla de fallos, comprobar que coincide con la de KMP y que las dos búsquedas dan las mismas posiciones, y comparar tiempos en un texto grande. En Python la búsqueda ingenua suele ganar en textos aleatorios, porque compara cortes de cadena en C y casi siempre falla al primer carácter; lo que aportan Z y KMP es el coste $O(n + m)$ garantizado en el peor caso.

In [5]:
def tabla_fallos(patron):
    fallos, k = [0] * len(patron), 0
    for i in range(1, len(patron)):
        while k and patron[i] != patron[k]:
            k = fallos[k - 1]
        if patron[i] == patron[k]:
            k += 1
        fallos[i] = k
    return fallos

def kmp(texto, patron):
    fallos, k, posiciones = tabla_fallos(patron), 0, []
    for i, c in enumerate(texto):
        while k and c != patron[k]:
            k = fallos[k - 1]
        if c == patron[k]:
            k += 1
        if k == len(patron):
            posiciones.append(i - k + 1)
            k = fallos[k - 1]
    return posiciones

def fallos_desde_z(s):
    z, fallos = array_z(s), [0] * len(s)
    for i in range(1, len(s)):
        # de derecha a izquierda; si ya está fijado, lo de la izquierda también
        for j in range(z[i] - 1, -1, -1):
            if fallos[i + j]:
                break
            fallos[i + j] = j + 1
    return fallos

for _ in range(1000):
    t = "".join(random.choices("ab", k=random.randint(1, 20)))
    assert fallos_desde_z(t) == tabla_fallos(t)
    p = "".join(random.choices("ab", k=random.randint(1, 4)))
    assert busca_z(t, p) == kmp(t, p)
print(array_z("ABABCABAB"), fallos_desde_z("ABABCABAB"), tabla_fallos("ABABCABAB"))

random.seed(2)
texto = "".join(random.choices("ab", k=300_000))
patron = "abbabaab"
assert busca_z(texto, patron) == kmp(texto, patron) == ingenua(texto, patron)
for f in (busca_z, kmp, ingenua):
    print(f"{f.__name__:8} {timeit.timeit(lambda: f(texto, patron), number=1):.4f} s")

[9, 0, 2, 0, 0, 4, 0, 2, 0] [0, 0, 1, 2, 0, 1, 2, 3, 4] [0, 0, 1, 2, 0, 1, 2, 3, 4]
busca_z  0.0636 s
kmp      0.0222 s
ingenua  0.0177 s
